# URBAN FLOOD STRESS | Streets and Graph Metrics

Lee la red LION local y construye métricas de grafo dirigido.

Flujo:
- leer calles desde `lion.gdb`
- construir métricas de grafo
- guardar salidas en `data/spatial/vector/streets/processed/`

Requisitos:
- `geopandas`
- `python-igraph`


Notas breves:
- No descarga nada: usa `data/spatial/vector/streets/raw/lion.gdb`, copiado desde `/home/map10194/Documents/ml4c/ml4c-pro`.
- Si falta `python-igraph`, el notebook se detendrá con un error claro.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import geopandas as gpd


def find_project_root() -> Path:
    """Locate the repository root from the current notebook."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    return current


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from project_name.street_network import (
        compute_directed_graph_metrics,
        load_lion_street_layer,
    )
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'Este notebook requiere geopandas y python-igraph antes de ejecutar.'
    ) from exc

PROCESSED_DIR = ROOT / 'data' / 'spatial' / 'vector' / 'streets' / 'processed'
LION_SOURCE = ROOT / 'data' / 'spatial' / 'vector' / 'streets' / 'raw' / 'lion.gdb'
LION_LAYER = 'lion'
SNAP_TOLERANCE_FT = 5.0

if not LION_SOURCE.exists():
    raise FileNotFoundError(f'LION source not found: {LION_SOURCE}')

print(ROOT)
print(LION_SOURCE)
print(PROCESSED_DIR)


/home/map10194/Documents/urban-flood-stress
/home/map10194/Documents/urban-flood-stress/data/spatial/vector/streets/raw/lion.gdb
/home/map10194/Documents/urban-flood-stress/data/spatial/vector/streets/processed


## 1. Carga del LION local

Esto lee el GDB local y deja la red lista para las métricas.


In [2]:
streets_source = load_lion_street_layer(LION_SOURCE, layer=LION_LAYER)
print(f'rows: {len(streets_source)}')
print(f'crs: {streets_source.crs}')
print(LION_SOURCE)


rows: 242344
crs: EPSG:2263
/home/map10194/Documents/urban-flood-stress/data/spatial/vector/streets/raw/lion.gdb


## 2. Grafo dirigido y métricas

Aquí se calcula:
- métricas por segmento
- métricas por nodo
- métricas por arista


In [3]:
streets = streets_source.copy()
metrics, node_metrics, edge_metrics = compute_directed_graph_metrics(
    streets,
    snap_tolerance_ft=SNAP_TOLERANCE_FT,
)

metrics_out = PROCESSED_DIR / 'lion_metrics.gpkg'
node_out = PROCESSED_DIR / 'lion_node_metrics.csv'
edge_out = PROCESSED_DIR / 'lion_edge_metrics.csv'

metrics_gdf = gpd.GeoDataFrame(metrics, geometry='geometry', crs=streets.crs)
metrics_gdf.to_file(metrics_out, driver='GPKG')
node_metrics.to_csv(node_out, index=False)
edge_metrics.to_csv(edge_out, index=False)

print(f'street segments with metrics: {len(metrics_gdf)}')
print(f'graph nodes: {len(node_metrics)}')
print(f'graph edges: {len(edge_metrics)}')
print(metrics_out)
print(node_out)
print(edge_out)


street segments with metrics: 196904
graph nodes: 134270
graph edges: 196904
/home/map10194/Documents/urban-flood-stress/data/spatial/vector/streets/processed/lion_metrics.gpkg
/home/map10194/Documents/urban-flood-stress/data/spatial/vector/streets/processed/lion_node_metrics.csv
/home/map10194/Documents/urban-flood-stress/data/spatial/vector/streets/processed/lion_edge_metrics.csv


In [4]:
metrics_gdf.head()

,Street,SAFStreetName,FeatureTyp,SegmentTyp,IncExFlag,RB_Layer,NonPed,TrafDir,TrafSrc,SpecAddr,...,length,travel_time,trafdir_resolved,trafdir_fallback_used,u,v,snapping_applied,snap_tolerance_ft,in_graph,edge_betweenness
0,EAST 168 STREET,,0,U,,B,,T,DOT,,...,120.710233,10.800844,T,False,0,1,True,5.0,True,4185041.0
1,WEST 192 STREET,,0,U,,B,,A,DOT,,...,85.149085,7.618923,A,False,2,3,True,5.0,True,2032651.0
2,UNION AVENUE,,0,U,,B,,W,DOT,,...,188.466110,16.863467,W,False,4,5,True,5.0,True,260185.0
3,UNION AVENUE,BEHAGEN PLAYGROUND COMFORT STA,0,U,,B,,W,DOT,X,...,188.466110,16.863467,W,False,4,5,True,5.0,True,260185.0
4,UNION AVENUE,BEHAGEN PLAYGROUND FIELD NORTH,0,U,,B,,W,DOT,X,...,188.466110,16.863467,W,False,4,5,True,5.0,True,260185.0
